# MST-GNN Paper Replication — Kaggle Notebook

> **Paper**: "Graph Representation Learning of Multilayer Spatial-Temporal Networks for Stock Predictions" (IEEE TCSS, 2024)

## Setup checklist
Before running: **Settings → Accelerator → GPU T4 x2** (right panel)

## Pipeline
```
Phase 1: Data collection  (~1–5 min, cached after first run)
Phase 2: Preprocessing    (~15 sec)
Phase 3: Graph building   (~10–20 min, 1088 trading days × 4 networks)
Phase 4: Dataset creation (~2 min)
Phase 5: Training         (~30–45 min on T4 GPU, 200 epochs)
Phase 6: Backtest         (~1 min)
Total:                    ~50–75 min
```

## Step 1 — Verify GPU

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: GPU not available — go to Settings → Accelerator → GPU T4 x2")

## Step 2 — Clone Repository

In [ ]:
import os

REPO_URL  = "https://github.com/quocnguyen5/mst-gnn-impl.git"
WORK_DIR  = "/kaggle/working/mst-gnn-impl"

if os.path.exists(WORK_DIR):
    # Pull latest changes if repo already exists
    print("Repo exists — pulling latest changes...")
    os.system(f"git -C {WORK_DIR} pull")
else:
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {WORK_DIR}")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir("."))

## Step 3 — Install Dependencies

In [ ]:
import torch, subprocess, sys

# Core packages
!pip install -q akshare baostock gensim jieba tqdm pyarrow fastparquet

# Detect GPU compute capability and pick the right PyTorch + PyG version
pyt = torch.__version__.split('+')[0]
cuda_str = torch.version.cuda or 'cpu'
cuda_tag = cuda_str.replace('.', '')

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)  # e.g. (6, 0) for P100
    cap_int = cap[0] * 10 + cap[1]              # e.g. 60 for sm_60
    print(f"GPU compute capability: sm_{cap_int}")

    if cap_int < 70:
        # P100 (sm_60) — needs PyTorch <= 2.1.x
        print("P100 detected (sm_60) — installing PyTorch 2.1.2 (last version supporting sm_60)")
        !pip install -q torch==2.1.2+cu121 torchvision==0.16.2+cu121 \
            --index-url https://download.pytorch.org/whl/cu121 \
            --force-reinstall -q
        pyg_torch = '2.1.2'
        pyg_cuda  = 'cu121'
    else:
        # T4/V100/A100 (sm_70+) — use current PyTorch
        print(f"GPU sm_{cap_int} — using installed PyTorch {pyt}")
        pyg_torch = pyt
        pyg_cuda  = f'cu{cuda_tag}'
else:
    print("No GPU — using CPU")
    pyg_torch = pyt
    pyg_cuda  = 'cpu'

print(f"Installing PyG for torch-{pyg_torch}+{pyg_cuda}...")
!pip install -q torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-{pyg_torch}+{pyg_cuda}.html
!pip install -q torch-geometric

import importlib
import torch_geometric
print(f"torch-geometric: {torch_geometric.__version__}")
print("\nNOTE: If PyTorch was downgraded, restart the kernel before proceeding!")
import torch
print(f"Active PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")


## Step 4 — Sanity Check (Optional)
Run a quick 3-epoch test with synthetic data to verify the model works before the full run.

In [ ]:
# Quick sanity check — synthetic data, 3 epochs, ~30 seconds on GPU
# Skip this cell if you want to go straight to the real experiment
import subprocess, sys
proc = subprocess.Popen(
    [sys.executable, "-m", "experiments.run_sanity_check"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd="/kaggle/working/mst-gnn-impl"
)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.wait()
print("Sanity check complete!" if proc.returncode == 0 else f"Exit code: {proc.returncode}")

## Step 5 — Run Main Experiment (CSI 300)

Full pipeline: data collection → preprocessing → graph building → training → backtest.

**Expected output during run:**
```
[Phase 1] Collecting data...        ~1-5 min
[Phase 2] Preprocessing...          ~15 sec
[Phase 3] Building graphs...        ~10-20 min  ← progress bar visible
[Phase 4] Creating dataset...       ~2 min
[Phase 5] Training MST-GNN...       ~30-45 min  ← epoch logs every 10 epochs
[Phase 6] Backtest...               ~1 min
```

In [ ]:
# Run CSI 300 with mean aggregator — standard variant, fastest
# Output streams line-by-line for real-time progress monitoring
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-m", "experiments.run_main",
     "--dataset", "csi300",
     "--aggregator", "mean"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,   # merge stderr so all output is visible
    text=True, bufsize=1,
    cwd="/kaggle/working/mst-gnn-impl"
)

for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)

proc.wait()
print(f"\nProcess finished — return code: {proc.returncode}")

## Step 6 — Inspect Results
Verify snapshot count and display the backtest chart after training completes.

In [ ]:
# Check cached snapshot count after training
import pickle, os

for dataset in ["csi300", "csi500"]:
    pkl = f"data/processed/{dataset}_snapshots.pkl"
    if os.path.exists(pkl):
        with open(pkl, "rb") as f:
            snaps = pickle.load(f)
        n = len(snaps)
        print(f"{dataset.upper():7s}: {n} snapshots "
              f"(train~{int(n*0.7)}, val~{int(n*0.1)}, test~{int(n*0.2)})")
    else:
        print(f"{dataset.upper():7s}: not built yet")

In [ ]:
# Display saved backtest chart
from IPython.display import Image, display
import os

for dataset in ["csi300", "csi500"]:
    chart = f"checkpoints/cumulative_returns_{dataset}.png"
    if os.path.exists(chart):
        print(f"--- {dataset.upper()} Cumulative Returns ---")
        display(Image(filename=chart))
    else:
        print(f"{dataset.upper()}: no chart yet (run experiment first)")

### Step 6b — Extract Test Metrics from Log File

Reads the training log and prints the final **Accuracy / Precision / DAMRR** table.
These are the numbers to report in your paper presentation.

In [ ]:
# Extract test metrics from log files — run after Step 5 completes
import glob, re, os

def parse_metrics_from_log(log_path):
    """Parse Accuracy, Precision, DAMRR from a training log file."""
    metrics = {}
    with open(log_path) as f:
        lines = f.readlines()
    in_test = False
    for line in lines:
        if 'TEST RESULTS' in line:
            in_test = True
        if in_test:
            for key in ['Accuracy', 'Precision', 'DAMRR', 'Loss']:
                m = re.search(rf'{key}:\s*([0-9.]+)', line)
                if m:
                    metrics[key] = float(m.group(1))
        if in_test and 'Loss' in metrics and len(metrics) >= 4:
            break
    return metrics

log_dir = '/kaggle/working/mst-gnn-impl/logs'
log_files = sorted(glob.glob(f'{log_dir}/**/*.log', recursive=True) +
                   glob.glob(f'{log_dir}/*.log'))

if not log_files:
    print('No log files found. Run Step 5 first.')
else:
    print(f'Found {len(log_files)} log file(s)\n')
    print(f'{"Model":<30} {"Accuracy":>10} {"Precision":>10} {"DAMRR":>10} {"Loss":>10}')
    print('-' * 65)
    for log in log_files:
        name = os.path.basename(os.path.dirname(log)) or os.path.basename(log)
        m = parse_metrics_from_log(log)
        if m:
            print(f'{name:<30} {m.get("Accuracy",0):>10.4f} '
                  f'{m.get("Precision",0):>10.4f} '
                  f'{m.get("DAMRR",0):>10.4f} '
                  f'{m.get("Loss",0):>10.4f}')
        else:
            print(f'{name:<30} (no TEST RESULTS found — training may still be running)')


### Step 6c — Epoch Learning Curve

Plots train/val loss and accuracy over epochs — shows the model is learning and
converging (not overfitting).

In [ ]:
# Plot training curves (loss + accuracy) from log file
import glob, re, os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def parse_epoch_logs(log_path):
    """Parse per-epoch train/val metrics from log."""
    epochs, train_loss, val_loss = [], [], []
    train_acc, val_acc = [], []
    # Pattern: Epoch  10/200 (2.3s) | LR: ... | Train Loss: X Acc: Y ... | Val Loss: X Acc: Y ...
    pattern = re.compile(
        r'Epoch\s+(\d+)/\d+.*?'
        r'Train Loss:\s*([0-9.]+)\s+Acc:\s*([0-9.]+).*?'
        r'Val Loss:\s*([0-9.]+)\s+Acc:\s*([0-9.]+)'
    )
    with open(log_path) as f:
        for line in f:
            m = pattern.search(line)
            if m:
                epochs.append(int(m.group(1)))
                train_loss.append(float(m.group(2)))
                train_acc.append(float(m.group(3)))
                val_loss.append(float(m.group(4)))
                val_acc.append(float(m.group(5)))
    return epochs, train_loss, val_loss, train_acc, val_acc

log_dir = '/kaggle/working/mst-gnn-impl/logs'
log_files = sorted(glob.glob(f'{log_dir}/**/*.log', recursive=True) +
                   glob.glob(f'{log_dir}/*.log'))

if not log_files:
    print('No log files found. Run Step 5 first.')
else:
    log = log_files[0]  # use first log (main experiment)
    epochs, tl, vl, ta, va = parse_epoch_logs(log)

    if not epochs:
        print('No epoch data found in log. Training may still be running.')
    else:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle('MST-GNN Training Curves (CSI 300)', fontsize=14, fontweight='bold')

        # Loss curve
        ax1.plot(epochs, tl, label='Train Loss', color='#2196F3', linewidth=1.5)
        ax1.plot(epochs, vl, label='Val Loss',   color='#FF5722', linewidth=1.5, linestyle='--')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title('Loss over Epochs')
        ax1.legend()
        ax1.grid(alpha=0.3)

        # Accuracy curve
        ax2.plot(epochs, ta, label='Train Acc', color='#4CAF50', linewidth=1.5)
        ax2.plot(epochs, va, label='Val Acc',   color='#FF9800', linewidth=1.5, linestyle='--')
        ax2.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, label='Random (0.5)')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.set_title('Accuracy over Epochs')
        ax2.set_ylim(0.3, 0.8)
        ax2.legend()
        ax2.grid(alpha=0.3)

        plt.tight_layout()
        save_path = '/kaggle/working/mst-gnn-impl/checkpoints/training_curves.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved to {save_path}')
        print(f'Total epochs logged: {len(epochs)}')
        print(f'Best val accuracy:   {max(va):.4f} (epoch {epochs[va.index(max(va))]})')
        print(f'Final train acc:     {ta[-1]:.4f}')
        print(f'Final val acc:       {va[-1]:.4f}')


### Step 6d — Full Results Summary Table

Aggregates all results into a single presentation-ready table.
Compare across aggregator variants if ablation (Step 7) was run.

In [ ]:
# Generate a complete results summary — ready to copy into your slides
import glob, re, os
import pandas as pd

def extract_all_metrics(log_dir):
    """Extract metrics from all log files and return as DataFrame."""
    rows = []
    log_files = sorted(glob.glob(f'{log_dir}/**/*.log', recursive=True) +
                       glob.glob(f'{log_dir}/*.log'))
    for log in log_files:
        with open(log) as f:
            content = f.read()
        # Extract experiment name from log header or filename
        name_m = re.search(r'MST-GNN Experiment: (\S+) with (\S+) aggregator', content)
        if name_m:
            dataset = name_m.group(1)
            agg     = name_m.group(2)
            model   = f'MST-GNN-{agg}'
        else:
            model   = os.path.basename(log).replace('.log', '')
            dataset = 'csi300'

        # Extract test metrics
        acc_m  = re.search(r'Accuracy:\s*([0-9.]+)', content[content.find('TEST'):] if 'TEST' in content else '')
        prec_m = re.search(r'Precision:\s*([0-9.]+)', content[content.find('TEST'):] if 'TEST' in content else '')
        damrr_m= re.search(r'DAMRR:\s*([0-9.]+)',    content[content.find('TEST'):] if 'TEST' in content else '')

        if acc_m:
            rows.append({
                'Model': model,
                'Dataset': dataset.upper(),
                'Accuracy': float(acc_m.group(1)),
                'Precision': float(prec_m.group(1)) if prec_m else None,
                'DAMRR': float(damrr_m.group(1)) if damrr_m else None,
            })
    return pd.DataFrame(rows)

log_dir = '/kaggle/working/mst-gnn-impl/logs'
df = extract_all_metrics(log_dir)

if df.empty:
    print('No completed experiments found. Run Step 5 (and optionally Step 7) first.')
else:
    # Sort: best accuracy first
    df = df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

    print('=' * 65)
    print('          MST-GNN RESULTS SUMMARY (CSI 300 Test Set)')
    print('=' * 65)
    print(df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    print('=' * 65)
    print()
    print('Paper baseline (Table IV, MST-GNN-mean):')
    print('  Accuracy:  0.5873')
    print('  Precision: 0.6012')
    print('  DAMRR:     0.5741')
    print()
    if len(df) > 0:
        best = df.iloc[0]
        paper_acc = 0.5873
        diff = best['Accuracy'] - paper_acc
        print(f'Your best result vs paper: {diff:+.4f} ({"✅ matches" if abs(diff) < 0.02 else "⚠️ check"})')
        print('(Difference < 0.02 is within expected variance from random seed + empty shareholding/topicality networks)')
    display(df.style.highlight_max(subset=['Accuracy','Precision','DAMRR'], color='#d4edda'))


## Step 7 — Ablation Study (Optional)
Compare all 3 STNA aggregator variants: mean, lstm, maxpool.
Reproduces Table VI from the paper.

> **Estimated time**: ~2–3 hours (3 full training runs)

In [ ]:
# Run all 3 aggregator variants and print a comparison table
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-m", "experiments.run_main",
     "--dataset", "csi300",
     "--aggregator", "all"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd="/kaggle/working/mst-gnn-impl"
)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.wait()

## Step 8 — Vietnam Market Extension (VN30 / VN100)

Apply MST-GNN to Vietnamese stocks (HOSE exchange) using the `vnstock` library.

**Active networks:**
| Network | Status |
|---|---|
| Comovement | ✅ Active (price correlation) |
| Industry | ✅ Active (ICB classification via vnstock) |
| Shareholding | ⚪ Empty (future work) |
| Topicality | ⚪ Empty (requires Vietnamese NLP — future work) |

In [ ]:
# Install vnstock — free Vietnam stock data (works globally, no API key)
!pip install -q vnstock
print("vnstock installed.")

In [ ]:
# Run MST-GNN on VN30 — fastest Vietnam experiment (~30-45 min)
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-m", "experiments.run_vn",
     "--universe", "vn30",
     "--aggregator", "mean"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd="/kaggle/working/mst-gnn-impl"
)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.wait()
print(f"\nVN30 experiment done — return code: {proc.returncode}")

In [ ]:
# Run MST-GNN on VN Top 100 — recommended for lab presentation (~60-90 min)
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-m", "experiments.run_vn",
     "--universe", "vn100",
     "--aggregator", "mean"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd="/kaggle/working/mst-gnn-impl"
)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.wait()
print(f"\nVN100 experiment done — return code: {proc.returncode}")

In [ ]:
# Inspect VN results
import pickle, os
from IPython.display import Image, display

print("=== Vietnam Dataset Status ===")
for universe in ["vn30", "vn100"]:
    pkl = f"data/processed_vn/{universe}_snapshots.pkl"
    if os.path.exists(pkl):
        with open(pkl, "rb") as f:
            snaps = pickle.load(f)
        n = len(snaps)
        print(f"{universe.upper():6s}: {n} snapshots "
              f"(train~{int(n*0.7)}, val~{int(n*0.1)}, test~{int(n*0.2)})")
    else:
        print(f"{universe.upper():6s}: not built yet")

# Display charts
for universe in ["vn30", "vn100"]:
    chart = f"checkpoints/cumulative_returns_{universe}.png"
    if os.path.exists(chart):
        print(f"\n--- {universe.upper()} Cumulative Returns ---")
        display(Image(filename=chart))